# Counter-Ryu · Windows / WSL2 工作入口

先读仓库根目录 `AGENTS.md` 和 `initial_readme_1022.md`。本 notebook 调用同一组 Python 模块，不复制训练代码。原课程 notebook 只读，不直接 Run All。禁止文件哈希和原课程打包/校验路径。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "initial_readme_1022.md").is_file())
os.chdir(ROOT)
os.environ["PYTHONUTF8"] = "1"
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"

def run(module, *arguments):
    subprocess.run([sys.executable, "-m", module, *map(str, arguments)], cwd=ROOT, check=True)

print("Repository:", ROOT)
print("Python:", sys.executable)


## 1. 无需 ROM 的环境与固定输入检查

应比较 80 个合法观察，动作差异为 0。失败先检查环境和加载，不直接训练。


In [ ]:
run("scripts.check_environment")
run("scripts.check_inference")


## 2. 安装明确指定的本地 ROM，再检查模拟器

先拉取最新 Git 主干，课程 ROM 已在 local/ 目录。若 integration 已安装 ROM，无需重装。以下开关默认关闭；只检查明确的本地输入，不扫描或计算文件摘要。


In [ ]:
INSTALL_ROM = False
ROM_PATH = ROOT / "local/StreetFighterIISpecialChampionEdition-Genesis.md"
if INSTALL_ROM:
    run("scripts.install_local_rom", "--rom", ROM_PATH)

RUN_NATIVE_CHECK = False
if RUN_NATIVE_CHECK:
    run("scripts.check_native", "--out", "artifacts/windows_native_check.json")


## 3. 完整困难基线（24 局 PvP + 8 局 PvE）

历史参考 11/24、8/8；这是反复使用的困难开发集，不是未知对手保证。输出目录必须是新的。`arena-call-sequence` 仅模拟调用序列，不是官方 Arena 验收。


In [ ]:
RUN_FOCUS_EVALUATION = False
if RUN_FOCUS_EVALUATION:
    run("experiments.league_adaptive.evaluate",
        "--agent", "models/counter_ryu_final/agent.pt",
        "--pool", "configs/focus_pool.json",
        "--out", "artifacts/windows_reference_focus",
        "--arena-call-sequence")


## 4. 按需录制完整对局

观察实际位移和交互收益，不把走路次数当作战术能力。视频留在本机，不自动上传。


In [ ]:
RECORD_PVE8 = False
if RECORD_PVE8:
    run("experiments.league_adaptive.record",
        "--agent", "models/counter_ryu_final/agent.pt",
        "--cpu-character", "Ryu", "--cpu-stars", "8",
        "--out", "artifacts/video_pve8", "--label", "Counter-Ryu")


## 5. 可选普通 PPO 短运行（初次交接不启动）

此入口保持现有 mixture 架构，是普通 PPO 续训冒烟，不承诺 adaptive 提升。先复现并审核，再明确开启。参考权重不会被覆盖，训练完必须另做冻结整局评估。


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    run("scripts.train_reference_ppo", "--steps", "3072",
        "--max-seconds", "180", "--out", "artifacts/windows_ppo_smoke")


## 6. 回传 Mac

依照 `initial_readme_1022.md` 在任务分支提交必要源码、配置和精选 JSON 结果。不要提交 notebook 输出、大量权重/录像、环境、其他 ROM 或整个 artifacts；课程匹配的单一 ROM 已由用户要求纳入 Git。Prompt 由聊天转交，不写入仓库；不要新建一串 Markdown。
